<a href="https://colab.research.google.com/github/sachinshekar35-stack/java1/blob/main/Company_Knowledge_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install -U google-genai sentence-transformers faiss-cpu pypdf gradio

In [12]:
import os
import getpass

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("✅ API key loaded")

Enter your Gemini API key: ··········
✅ API key loaded


In [14]:
from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain RAG in one simple sentence."
)

print(response.text)

**RAG (Retrieval-Augmented Generation)** is an AI technique that improves answers by allowing a language model to look up relevant facts from an external database before generating a response—like giving the AI an open-book exam.


In [10]:
company_policy = """
ABC Technologies Employee Handbook

WORKING HOURS

Employees normally work from Monday to Friday.
Standard working hours are 9:00 AM to 6:00 PM.

ANNUAL LEAVE

Employees receive 20 days of annual paid leave per year.

SICK LEAVE

Employees may request sick leave through the HR portal.
A medical certificate may be required for extended sick leave.

WORK FROM HOME

Employees can work from home up to 2 days per week,
subject to manager approval.

REIMBURSEMENT

Employees can request reimbursement for approved business
expenses through the finance portal.

IT SECURITY

Employees must not share company passwords.
Company laptops must use the approved security software.

EMPLOYEE SUPPORT

For HR questions, employees should contact the HR department.
For technical problems, employees should contact the IT help desk.
"""

with open("/content/company_policy.txt", "w") as f:
    f.write(company_policy)

print("Created /content/company_policy.txt")

Created /content/company_policy.txt


In [15]:
import os
import re
import time
import faiss
import numpy as np
import gradio as gr

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai


# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = os.environ["GEMINI_API_KEY"]

# Current stable Gemini model
GEMINI_MODEL = "gemini-3.6-flash"

# Create Gemini client
client = genai.Client(
    api_key=API_KEY
)

# Embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# GLOBAL VARIABLES
# ============================================================

chunks = []
vector_database = None


# ============================================================
# READ PDF
# ============================================================

def read_pdf(file_path):

    text = ""

    reader = PdfReader(file_path)

    for page_number, page in enumerate(reader.pages):

        page_text = page.extract_text()

        if page_text:

            text += (
                f"\n[Page {page_number + 1}]\n"
                f"{page_text}\n"
            )

    return text


# ============================================================
# READ TXT
# ============================================================

def read_txt(file_path):

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        return file.read()


# ============================================================
# READ DOCUMENT
# ============================================================

def read_document(file_path):

    filename = os.path.basename(file_path)

    if filename.lower().endswith(".pdf"):

        return read_pdf(file_path)

    elif filename.lower().endswith(".txt"):

        return read_txt(file_path)

    else:

        return ""


# ============================================================
# CLEAN TEXT
# ============================================================

def clean_text(text):

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# CHUNK TEXT
# ============================================================

def create_chunks(
    text,
    chunk_size=500,
    overlap=80
):

    words = text.split()

    chunks_list = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        if chunk.strip():

            chunks_list.append(
                chunk
            )

        start += (
            chunk_size - overlap
        )

    return chunks_list


# ============================================================
# BUILD VECTOR DATABASE
# ============================================================

def build_knowledge_base(files):

    global chunks
    global vector_database

    # Reset old database
    chunks = []
    vector_database = None

    if not files:

        return (
            "❌ Please upload a PDF or TXT file."
        )

    total_documents = 0

    for file_path in files:

        try:

            filename = os.path.basename(
                file_path
            )

            print(
                f"Reading: {filename}"
            )

            text = read_document(
                file_path
            )

            text = clean_text(
                text
            )

            if not text:

                print(
                    f"⚠️ No text found: {filename}"
                )

                continue

            document_chunks = create_chunks(
                text
            )

            for chunk in document_chunks:

                chunks.append({
                    "text": chunk,
                    "source": filename
                })

            total_documents += 1

        except Exception as error:

            print(
                f"Error reading file: {error}"
            )


    # Check documents
    if not chunks:

        return (
            "❌ No readable text was found "
            "in your uploaded files."
        )


    # ========================================================
    # CREATE EMBEDDINGS
    # ========================================================

    print(
        "Creating embeddings..."
    )

    texts = [
        item["text"]
        for item in chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True
    )


    # ========================================================
    # NORMALIZE
    # ========================================================

    faiss.normalize_L2(
        embeddings
    )


    # ========================================================
    # CREATE FAISS INDEX
    # ========================================================

    dimension = embeddings.shape[1]

    vector_database = faiss.IndexFlatIP(
        dimension
    )

    vector_database.add(
        embeddings
    )


    return f"""
# ✅ Knowledge Base Ready

📄 Documents processed: **{total_documents}**

🧩 Text chunks: **{len(chunks)}**

🔢 Embedding dimension: **{dimension}**

🤖 Gemini model: **{GEMINI_MODEL}**

You can now ask questions.
"""


# ============================================================
# RETRIEVE RELEVANT DOCUMENTS
# ============================================================

def retrieve_documents(
    question,
    top_k=5
):

    if vector_database is None:

        return []

    # Embed question
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    # Normalize
    faiss.normalize_L2(
        question_embedding
    )

    # Search
    number_to_return = min(
        top_k,
        len(chunks)
    )

    scores, indexes = vector_database.search(
        question_embedding,
        number_to_return
    )

    results = []

    for score, index in zip(
        scores[0],
        indexes[0]
    ):

        if index < 0:
            continue

        results.append({
            "text": chunks[index]["text"],
            "source": chunks[index]["source"],
            "score": float(score)
        })

    return results


# ============================================================
# GEMINI WITH RETRY
# ============================================================

def generate_with_retry(
    prompt,
    max_attempts=5
):

    for attempt in range(
        max_attempts
    ):

        try:

            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )

            if response.text:

                return response.text

            return "Gemini returned an empty response."

        except Exception as error:

            error_message = str(error)

            print(
                f"Gemini attempt "
                f"{attempt + 1} failed:"
            )

            print(error_message)


            # Handle temporary server overload
            if (
                "503" in error_message
                or
                "UNAVAILABLE" in error_message
            ):

                if attempt < max_attempts - 1:

                    wait_seconds = (
                        2 ** attempt
                    )

                    print(
                        f"Retrying in "
                        f"{wait_seconds} seconds..."
                    )

                    time.sleep(
                        wait_seconds
                    )

                    continue


            # Other errors
            return f"""
❌ Gemini Error

{error_message}

Possible causes:

• API key problem
• Gemini service temporarily unavailable
• API quota/rate limit
• Model access problem

Please check the error above.
"""


    return """
❌ Gemini is temporarily unavailable.

Please wait a little and try again.
"""


# ============================================================
# ASK QUESTION
# ============================================================

def answer_question(
    question
):

    # Check database
    if vector_database is None:

        return """
⚠️ **Knowledge base not ready.**

Please:

1. Upload your company document.
2. Click **Build Knowledge Base**.
3. Ask your question.
"""


    # Check question
    if not question.strip():

        return (
            "Please enter a question."
        )


    # ========================================================
    # RETRIEVE
    # ========================================================

    results = retrieve_documents(
        question,
        top_k=5
    )


    if not results:

        return (
            "I couldn't find relevant "
            "information."
        )


    # ========================================================
    # BUILD CONTEXT
    # ========================================================

    context = ""

    for number, result in enumerate(
        results,
        start=1
    ):

        context += f"""

SOURCE {number}
FILE: {result["source"]}

{result["text"]}

--------------------------------
"""


    # ========================================================
    # PROMPT
    # ========================================================

    prompt = f"""
You are an AI Company Knowledge Assistant.

Your task is to answer employee questions using
ONLY the company knowledge provided below.

IMPORTANT RULES:

1. Do not invent company policies.
2. Do not use outside knowledge.
3. If the answer is not in the knowledge base,
   say exactly:

"I couldn't find this information in the
company knowledge base."

4. Give a clear and simple answer.
5. Mention the source file at the end.
6. If multiple sources are relevant, mention them.

COMPANY KNOWLEDGE:

{context}


EMPLOYEE QUESTION:

{question}


ANSWER:
"""


    # ========================================================
    # GEMINI
    # ========================================================

    answer = generate_with_retry(
        prompt
    )

    return answer


# ============================================================
# BUILD GRADIO APP
# ============================================================

with gr.Blocks(
    title="Company Knowledge Bot"
) as app:

    gr.Markdown(
        """
# 🏢 Company Knowledge Bot

### RAG-powered company document assistant

Upload company documents and ask questions about:

📋 HR policies
🏖️ Leave policies
💻 IT policies
🏠 Work-from-home rules
💰 Expense policies
📚 Employee handbook
🏢 Company procedures
"""
    )


    # ========================================================
    # FILE UPLOAD
    # ========================================================

    files = gr.File(
        label="📁 Drop Company Files Here",
        file_count="multiple",
        file_types=[
            ".pdf",
            ".txt"
        ],
        type="filepath"
    )


    # ========================================================
    # BUILD BUTTON
    # ========================================================

    build_button = gr.Button(
        "🔨 Build Knowledge Base",
        variant="primary"
    )


    # ========================================================
    # STATUS
    # ========================================================

    status = gr.Markdown(
        value="Upload a document to begin."
    )


    build_button.click(
        fn=build_knowledge_base,
        inputs=files,
        outputs=status
    )


    # ========================================================
    # QUESTION
    # ========================================================

    gr.Markdown(
        """
## 💬 Ask the Company Bot

Try:

**How many annual leave days do employees receive?**

**Can employees work from home?**

**What are the working hours?**

**How do I request reimbursement?**

**What should I do if I lose my laptop?**
"""
    )


    question = gr.Textbox(
        label="Your Question",
        placeholder=(
            "Example: "
            "How many annual leave days do employees get?"
        ),
        lines=2
    )


    ask_button = gr.Button(
        "🤖 Ask Company Bot",
        variant="primary"
    )


    answer = gr.Markdown(
        label="Answer"
    )


    ask_button.click(
        fn=answer_question,
        inputs=question,
        outputs=answer
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://283dc26f1506f81465.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Reading: ABC_Technologies_Employee_Handbook (1).txt
Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Gemini attempt 1 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 1 seconds...
Gemini attempt 1 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 1 seconds...
Gemini attempt 2 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 2 seconds...
Gemini attempt 3 failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying in 4 seconds...
Keyboard interruptio